In [ ]:
# Leitura do arquivo

import pandas as pd

dados = pd.read_csv('../data/marketing_investimento.csv')
dados.head()

In [ ]:
# Verificação da quantidade de linhas presentes na base, nulos e data types
dados.info()

In [ ]:
# Analise exploratoria dos dados

# Analise de variáveis categórica

import plotly.express as px


# contagem de valores categóricos
px.histogram(dados, x='aderencia_investimento', text_auto=True, )
px.histogram(dados, x='inadimplencia', text_auto=True, color='aderencia_investimento', barmode='group')
px.histogram(dados, x='estado_civil', text_auto=True, color='aderencia_investimento', barmode='group')
px.histogram(dados, x='escolaridade', text_auto=True, color='aderencia_investimento', barmode='group')
px.histogram(dados, x='fez_emprestimo', text_auto=True, color='aderencia_investimento', barmode='group')

In [ ]:
# Analise de variáveis numericas

# contagem de valores numericos
px.box(dados,x='idade', color='aderencia_investimento')
px.box(dados,x='saldo', color='aderencia_investimento')
px.box(dados,x='tempo_ult_contato', color='aderencia_investimento')
px.box(dados,x='numero_contatos', color='aderencia_investimento')



In [ ]:
# Selecionando as variáveis para o modelo

x = dados.drop('aderencia_investimento', axis=1) #variaveis explicativas
y = dados['aderencia_investimento'] #variavel alvo

In [ ]:
# Transformando variáveis explicativas - One hot encoding

from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder 

colunas = x.columns
one_hot = make_column_transformer((
    OneHotEncoder(drop='if_binary'),
    ['estado_civil','escolaridade','inadimplencia','fez_emprestimo']
),
    remainder='passthrough',
    sparse_threshold=0 #colunas se mantenham nos mesmo valores
    )

In [ ]:
x = one_hot.fit_transform(x)

In [ ]:
one_hot.get_feature_names_out(colunas)

In [ ]:
pd.DataFrame(x, columns=one_hot.get_feature_names_out(colunas))

In [ ]:
# Transformando varivel alvo - LabelEncoder

from sklearn.preprocessing import LabelEncoder

label_ecoder = LabelEncoder()
y = label_ecoder.fit_transform(y) # TRansformando em 0 e 1

In [ ]:
pd.DataFrame(y)

In [ ]:
# Divisao do conjunto de dados de treino e teste

from sklearn.model_selection import train_test_split

x_treino, x_teste, y_treino, y_teste = train_test_split(x,y, stratify=y, random_state=5)

In [ ]:
# Modelo de base - Dummy - Classifica com base no valor de maior frequencia da variavel alvo
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier() #inicializa o modelo e a partir disso ele se ajusta nos dados com a variavel

dummy.fit(x_treino, y_treino) #funcao responsavel por aprender o padrão nos dados
dummy.score(x_teste, y_teste) #essa função calcula a taxa de acerto do modelo, ou seja, a proporção de classificações corretas que ele fez em relação ao total de classificações realizadas


In [ ]:
# Modelo de classificação - Arvore de decisão

from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

arvore = DecisionTreeClassifier(random_state=5) #Mesmo resultado

arvore.fit(x_treino ,y_treino) #funcao responsavel por treinar os dados
arvore.predict(x_teste) #previsoes de classificação da arvore com dados novos

arvore.score(x_teste, y_teste) #resultado de 66% das vezes


# Nomes mais visuais das colunas
nome_colunas = ['casado (a)',
                'divorciado (a)',
                'solteiro (a)',
                'fundamental',
                'medio',
                'superior',
                'inadimplencia',
                'fez_emprestimo',
                'idade',
                'saldo',
                'tempo_ult_contato',
                'numero_contatos']

plt.figure(figsize=(15,6))
plot_tree(arvore, filled=True, class_names=['nao','sim'], fontsize=1, feature_names=nome_colunas)

In [ ]:
arvore.score(x_treino, y_treino)

In [ ]:
arvore = DecisionTreeClassifier(random_state=5, max_depth=3) #Mesmo resultado

arvore.fit(x_treino ,y_treino) #funcao responsavel por treinar os dados

In [ ]:
arvore.score(x_treino, y_treino)
arvore.score(x_teste, y_teste)

In [ ]:
plt.figure(figsize=(15,6))
plot_tree(arvore, filled=True, class_names=['nao','sim'], fontsize=7, feature_names=nome_colunas)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

normalizacao = MinMaxScaler()
x_treino_normalizado = normalizacao.fit_transform(x_treino)

pd.DataFrame(x_treino_normalizado)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier() #valor 3 por padrão
knn.fit(x_treino_normalizado, y_treino)

x_teste_normalizado = normalizacao.transform(x_teste)

In [ ]:
knn.score(x_teste_normalizado, y_teste)

In [ ]:
print(f'Acuracia dummy: {dummy.score(x_teste, y_teste)}')
print(f'Acuracia arvore: {arvore.score(x_teste, y_teste)}') # Melhor modelo
print(f'Acuracia knn: {knn.score(x_teste_normalizado, y_teste)}')

In [ ]:
# Exportacao do modelo - Pickle

# Armazenar o modelo em um arquivo serializados
import pickle

with open('modelo_onehot.pkl','wb') as arquivo:
        pickle.dump(one_hot, arquivo)
        

In [ ]:
with open('modelo_arvore.pkl','wb') as arquivo:
        pickle.dump(arvore, arquivo)

In [ ]:
novo_dado = {
    'idade': [45],
    'estado_civil':['solteiro (a)'],
    'escolaridade':['superior'],
    'inadimplencia': ['nao'],
    'saldo': [23040],
    'fez_emprestimo': ['nao'],
    'tempo_ult_contato': [800],
    'numero_contatos': [4]
}

In [ ]:
novo_dado = pd.DataFrame(novo_dado)
novo_dado

In [ ]:
modelo_one_hot = pd.read_pickle('modelo_onehot.pkl')
modelo_arvore = pd.read_pickle('modelo_arvore.pkl')

In [ ]:
modelo_arvore.predict(novo_dado)

In [ ]:
novo_dado = modelo_one_hot.transform(novo_dado)

In [ ]:
modelo_arvore.predict(novo_dado)
